In [ ]:
# import dependencies

# for hidden variables
!pip install python-dotenv --user --no-warn-script-location
from dotenv import load_dotenv

# to build spark session
from pyspark.sql import SparkSession

# to import the data
import os
import requests
from datetime import datetime
from dateutil.relativedelta import relativedelta
import time
!pip install requests --user --no-warn-script-location # to capture the data from the URL's where the data is stored

# for spark operations
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, DoubleType, IntegerType

In [2]:
# hidden and predefined variables

# username
load_dotenv() 
user = os.getenv('user')

# the folder where the data will be stored in the directory
folder_name = 'taxi_data'

In [3]:
''' 
    Set up spark session builder with appropriate values to match the Jupyter environment variables.
    All values are in GB.
''' 

###################################################################################################
###################################################################################################
                       # SET THESE VALUES TO MATCH THE JUPYTER ENVIRONMENT #                        
###################################################################################################
###################################################################################################
# total cores
total_executor_cores = 8

# total memory/RAM allocated per node
total_memory = 64

# set driver memory to 1-2GB for testing and development, up to 4GB for production if necessary
driver_memory_reserve = 2

###################################################################################################
###################################################################################################
                        # THESE VALUES WILL BE CALCULATED AUTOMATICALLY #                        
###################################################################################################
###################################################################################################

# reserve 1 core for the driver and rest for executors
executor_cores = total_executor_cores - 1

# amount of memory allocated for each executor
executor_memory = int((total_memory - driver_memory_reserve) / executor_cores)

###################################################################################################
###################################################################################################
                                 # BUILDING THE SPARK SESSION #                        
###################################################################################################
###################################################################################################

spark = SparkSession.builder \
    .config("spark.driver.memory", f"{driver_memory_reserve}g") \
    .config("spark.executor.memory", f"{executor_memory}g") \
    .config('spark.executor.instances', executor_cores) \
    .getOrCreate()


In [4]:
# import data

# capture the base URL and create a local folder to store the files.
# the base URL will be appended with the dates for the files we want to capture below.
base_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_'
data_folder = f'/expanse/lustre/projects/uci157/{user}/nyc_taxi_surge_pricing/{folder_name}'

# if the folder doesn't exist, create it
os.makedirs(data_folder, exist_ok=True)

# create a function that will generate the date structure and range for the files we want to download
# start_date, end_date are the respective starting and ending dates in year format
# iterates through every month from the starting year to ending year
def url_date_gen(start_date, end_date):
    current_date = start_date
    while current_date <= end_date:
        yield current_date
        current_date += relativedelta(months=1)

# define the start and end dates
start_date = datetime(2019,2,1)
end_date = datetime(2026,2,1)

# file counter to check track if all files were downloaded already
file_counter = 0

# download the files that contain the data
for current_date in url_date_gen(start_date, end_date):
    
    # format the date to be in the format that the URL uses: YYYY-MM
    file_date = current_date.strftime('%Y-%m')
    file_name = f'High_Volume_Trip_Data_{file_date}.parquet'
    file_url = f'{base_url}{file_date}.parquet'
    file_path = os.path.join(data_folder, file_name)

    # check if the file exists before attempting download
    if os.path.exists(file_path):
        file_counter += 1 # add 1 if file is already available
    else:
        # use try and except to capture any issues
        # send a request to download the file
        try:
            print(f'Downloading... {file_name} from {file_url}...')
            response = requests.get(file_url)
            response.raise_for_status()
    
            # if successful, write the content to a file 
            # so it is saved to disk and not just in memory
            with open(file_path, 'wb') as file:
                file.write(response.content)
            print(f'Downloaded {file_name} successfully!')
    
            time.sleep(5) # sleep for 5 second between requests
    
        # if there are any issues, print an error message
        except requests.exceptions.RequestException as e:
            print(f'Failed to download {file_name}: {e}')

# if all files were already available, print a message to let user know
# otherwise say how many files were available and skipped
if file_counter == (end_date - start_date).days + 1:
    print('All files available. No additional files have been downloaded.')
else:
    print(f'{file_counter} files are already available and were skipped.')

85 files are already available and were skipped.


In [8]:
# read the files into a spark dataframe and check to see if the schema is correct
# it looks like airport_fee is being inferred as an integer when it should be a double
# and wav_match_flag is being inferred as an integer when it should be a string
df = spark.read.parquet(data_folder)
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_

In [18]:
# we want airport_fee to be double so we will cast it
# and also ensure the rest of the columns are correct.
# define the schema using StructType and StructField
schema = StructType([
    StructField("hvfhs_license_num", StringType(), True),
    StructField("dispatching_base_num", StringType(), True),
    StructField("originating_base_num", StringType(), True),
    StructField("request_datetime", TimestampType(), True),
    StructField("on_scene_datetime", TimestampType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropoff_datetime", TimestampType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("trip_miles", DoubleType(), True),
    StructField("trip_time", LongType(), True),
    StructField("base_passenger_fare", DoubleType(), True),
    StructField("tolls", DoubleType(), True),
    StructField("bcf", DoubleType(), True),
    StructField("sales_tax", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),
    StructField("tips", DoubleType(), True),
    StructField("driver_pay", DoubleType(), True),
    StructField("shared_request_flag", StringType(), True),
    StructField("shared_match_flag", StringType(), True),
    StructField("access_a_ride_flag", StringType(), True),
    StructField("wav_request_flag", StringType(), True),
    StructField("wav_match_flag", StringType(), True)
])

# now read in the data with the defined schema
df = spark.read.schema(schema).parquet(data_folder)
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp (nullable = true)
 |-- on_scene_datetime: timestamp (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_flag: string (nul

In [25]:
# show first 5 rows vertically for legibility
df.show(n=5, vertical=True)

-RECORD 0-----------------------------------
 hvfhs_license_num    | HV0003              
 dispatching_base_num | B03404              
 originating_base_num | B03404              
 request_datetime     | 2022-04-01 00:18:10 
 on_scene_datetime    | 2022-04-01 00:23:57 
 pickup_datetime      | 2022-04-01 00:24:06 
 dropoff_datetime     | 2022-04-01 00:38:16 
 PULocationID         | 230                 
 DOLocationID         | 137                 
 trip_miles           | 2.63                
 trip_time            | 850                 
 base_passenger_fare  | 15.44               
 tolls                | 0.0                 
 bcf                  | 0.46                
 sales_tax            | 1.37                
 congestion_surcharge | 2.75                
 airport_fee          | 0.0                 
 tips                 | 0.0                 
 driver_pay           | 12.43               
 shared_request_flag  | N                   
 shared_match_flag    | N                   
 access_a_

In [26]:
# run this cell after initializing SparkSession and loading in data, take a screenshot of the output

import requests
import pandas as pd

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
df['maxMemory_GB'] = (df['maxMemory'] / (1024**3)).round(2)
df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,8,1099746508,0,True,1.02
